# Soluzioni — Lezione 7: Semantica distribuzionale
[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuri-bizzoni/yuribizzoni/blob/main/ml-humanities/solutions/07_semantica_distribuzionale_soluzioni.ipynb)

In [1]:
# ⚙️ CELLA DI PREPARAZIONE — eseguitela per prima (Shift+Invio) e non preoccupatevi del codice.
# Cerca la cartella "data" del corso; se non la trova (per esempio su Google Colab),
# scarica il materiale del corso da GitHub. Funziona anche senza internet, se avete la cartella.
from pathlib import Path
import subprocess

def trova_dati():
    for p in [Path("../data"), Path("data"), Path("ml-humanities/data"), Path("yuribizzoni/ml-humanities/data")]:
        if (p / "lessico_felicita.csv").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuri-bizzoni/yuribizzoni"], check=True)
    return Path("yuribizzoni/ml-humanities/data")

DATI = trova_dati()
print("Dati trovati in:", DATI.resolve())

Dati trovati in: /home/user/yuribizzoni/ml-humanities/data


In [2]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

def leggi_testo(percorso):
    return "\n".join(r for r in percorso.read_text(encoding="utf-8").splitlines() if not r.startswith("#"))

def parole(testo):
    return re.findall(r"\w+", testo.lower())

tokens = []
for percorso in sorted((DATI / "testi").glob("*.txt")):
    tokens += parole(leggi_testo(percorso))
print(f"{len(tokens):,} parole nel corpus del corso")

602,519 parole nel corpus del corso


In [3]:
def costruisci_vettori(tokens, finestra=4, min_freq=5, dimensioni=150):
    """Semantica distribuzionale in 4 passi: conteggi → co-occorrenze → PPMI → SVD."""
    # 1. vocabolario: solo le parole abbastanza frequenti
    freq = Counter(tokens)
    vocab = sorted(w for w, n in freq.items() if n >= min_freq)
    indice = {w: i for i, w in enumerate(vocab)}
    ids = np.array([indice.get(w, -1) for w in tokens])
    # 2. co-occorrenze: quante volte la parola A compare entro `finestra` parole da B
    righe, colonne = [], []
    for d in range(1, finestra + 1):
        a, b = ids[:-d], ids[d:]
        ok = (a >= 0) & (b >= 0)
        righe += [a[ok], b[ok]]
        colonne += [b[ok], a[ok]]
    righe, colonne = np.concatenate(righe), np.concatenate(colonne)
    C = sparse.coo_matrix((np.ones(len(righe)), (righe, colonne)), shape=(len(vocab), len(vocab))).tocsr().tocoo()
    # 3. PPMI: quanto due parole compaiono insieme PIÙ del previsto dal caso (solo valori positivi)
    totale = C.sum()
    p_riga = np.asarray(C.sum(axis=1)).ravel()
    p_col = np.asarray(C.sum(axis=0)).ravel() ** 0.75
    p_col = p_col / p_col.sum() * totale
    pmi = np.log(C.data * totale / (p_riga[C.row] * p_col[C.col]))
    tieni = pmi > 0
    PPMI = sparse.csr_matrix((pmi[tieni], (C.row[tieni], C.col[tieni])), shape=C.shape)
    # 4. SVD: comprimiamo migliaia di colonne in `dimensioni` numeri per parola
    E = TruncatedSVD(dimensioni, random_state=0).fit_transform(PPMI)
    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
    return E, vocab, indice

def vicini(parola, n=10, E=None, vocab=None, indice=None):
    E, vocab, indice = (E, vocab, indice) if E is not None else (VETTORI, VOCAB, INDICE)
    if parola not in indice:
        return f"'{parola}' non è nel vocabolario (troppo rara o assente)"
    sim = E @ E[indice[parola]]
    return [(vocab[i], round(float(sim[i]), 2)) for i in np.argsort(-sim)[1:n + 1]]

def similarita(a, b):
    return float(VETTORI[INDICE[a]] @ VETTORI[INDICE[b]])

In [4]:
VETTORI, VOCAB, INDICE = costruisci_vettori(tokens)

## Esercizio 1 — I vostri vicini

In [5]:
for p in ["napoli", "luna", "pane"]:
    print(p, "→", [w for w, s in vicini(p, 10)])

napoli → ['porto', 'venuta', 'rimaner', 'tenuto', 'trovarmi', 'parlato', 'anderò', 'tempo', 'vado', 'discorso']
luna → ['sole', 'stelle', 'luce', 'notte', 'raggio', 'ove', 'cielo', 'silenziosa', 'chiarore', 'stella']
pane → ['isidoro', 'po', 'mangiare', 'pure', 'fare', 'vino', 'ecco', 'pezzo', 'c', 'gridare']


**Commento.** Tre casi diversi. *luna* ha vicini davvero **semantici** (*sole, stelle, luce, notte, cielo*): è frequente e usata
in contesti coerenti, soprattutto da Leopardi e Dante. *napoli* ha vicini **grammaticali**: verbi di movimento e di permanenza
(*venuta, rimaner, anderò, vado*), perché le città compaiono dopo *a, da, di*, come destinazioni. *pane* mescola il campo
semantico (*mangiare, vino, pezzo*) con un personaggio (*isidoro*, di Deledda): in un corpus piccolo il "significato" di una parola
dipende molto dal libro che la usa di più.

## Esercizio 2 — Un altro asse

In [6]:
def asse(positivi, negativi):
    v = np.mean([VETTORI[INDICE[p]] for p in positivi], axis=0) - np.mean([VETTORI[INDICE[p]] for p in negativi], axis=0)
    return v / np.linalg.norm(v)

lista = ["contadino", "signore", "donna", "uomo", "gioia", "dolore", "fame", "festa", "chiesa", "casa", "pane", "oro"]
for poli in [(["ricco", "ricchi", "signore", "oro"], ["povero", "poveri", "miseria", "fame"]),
             (["ricco", "ricchezza", "denari"], ["povero", "povertà", "miseria"])]:
    a = asse(*poli)
    proiezioni = pd.Series({p: float(VETTORI[INDICE[p]] @ a) for p in lista if p not in poli[0] + poli[1]}).sort_values()
    print("poli:", poli)
    print(proiezioni.round(3).to_string(), "\n")

poli: (['ricco', 'ricchi', 'signore', 'oro'], ['povero', 'poveri', 'miseria', 'fame'])
gioia       -0.217
dolore      -0.184
chiesa      -0.137
casa        -0.100
festa       -0.099
contadino   -0.073
pane        -0.045
donna       -0.033
uomo         0.007 

poli: (['ricco', 'ricchezza', 'denari'], ['povero', 'povertà', 'miseria'])
chiesa      -0.222
contadino   -0.182
casa        -0.161
uomo        -0.140
donna       -0.140
gioia       -0.130
dolore      -0.111
fame        -0.101
pane        -0.098
signore     -0.092
festa       -0.091
oro         -0.004 



**Commento.** Quasi tutte le parole cadono dalla parte di *povero*: il corpus (Pinocchio, i Promessi sposi, Deledda, Di Giacomo)
parla molto più di povertà che di ricchezza, quindi il polo "povero" ha contesti più ricchi e attira tutto. E cambiando le parole
dei poli l'ordine cambia (*gioia* passa dall'estremo al centro): il risultato è **instabile**. Lezione di metodo: prima di
interpretare un asse, verificate che non dipenda dalla scelta arbitraria di poche parole e dallo sbilanciamento del corpus.

## Esercizio 3 — La finestra

In [7]:
risultati = {}
for f in [2, 10]:
    E, voc, ind = costruisci_vettori(tokens, finestra=f)
    for p in ["gioia", "felicità"]:
        risultati[f"{p} (finestra {f})"] = [w for w, s in vicini(p, 10, E, voc, ind)]
pd.DataFrame(risultati)

,gioia (finestra 2),felicità (finestra 2),gioia (finestra 10),felicità (finestra 10)
0,storia,vera,tenerezza,affanni
1,dolcezza,famiglia,ritorno,mortal
2,famiglia,scoperta,pietà,beltà
3,profonda,gioia,dolore,lice
4,giornata,profonda,ribrezzo,fati
5,tristezza,età,provò,prole
6,grazia,disciplina,tanta,opre
7,nova,vita,rimorso,regni
8,infinita,speciale,provava,ahi
9,specie,natura,morte,natura


**Commento.** Le differenze ci sono, ma non seguono la regola in modo perfetto: con un corpus piccolo il rumore è alto.
Con la finestra 10 i vicini di *gioia* sono quasi tutti **sentimenti** (*tenerezza, pietà, dolore, rimorso, ribrezzo*) e i verbi
che li introducono (*provò, provava*): il "tema" emozione. Con la finestra 2 compaiono parole che condividono le **costruzioni
immediate** (*una ___ profonda, la ___ della famiglia, ___ infinita*) anche se il senso è lontano. Nessuna finestra è "giusta":
dipende da quale significato di *simile* ci interessa, e i risultati vanno sempre letti, non solo calcolati.